# Practical Pulsar Timing with PINT and JUG

In this tutorial we move from the *concepts* of pulsar timing to the *practice* of it:
building a timing model, fitting it to times of arrival (TOAs), and — crucially —
convincing ourselves that what is left over after the fit is consistent with our
noise model.

We will use two packages:

- [**PINT**](https://nanograv-pint.readthedocs.io/) — a mature, pure-Python timing
  package used across the IPTA. We will use it both for fitting and for
  *simulating* realistic TOAs.
- [**JUG**](https://github.com/MattTMiles/jug) — a fast, independent timing package
  built on [JAX](https://jax.readthedocs.io/), with automatic GPU acceleration,
  a generalised-least-squares fitter with full noise-model support, and an
  interactive GUI.

### What you will do

1. **Warm up** with PINT on a bundled example pulsar (load, plot residuals, fit).
2. **Solve a Shapiro delay** hidden in simulated data — and weigh a neutron star.
3. **Repeat the fit in JUG** and compare the two packages.
4. **Do it on real data**: reproduce the J1614−2230 two-solar-mass measurement.
5. **Drive it interactively** with the `jug-gui` interface.

This is the first of two notebooks; the second
(`timing_2_noise_gaussianity.ipynb`) covers timing in the presence of red and
DM noise, noise realisations, whitening, and Gaussianity testing.

All datasets in this notebook are either simulated on the fly or ship with PINT,
so the notebook is fully self-contained.

> This tutorial builds on the earlier libstempo/PINT introductions from previous
> schools — if you have not seen how individual timing parameters (F0, F1,
> position, parallax, proper motion) imprint themselves on residuals, those are a
> great companion exercise.


## 0. A one-minute recap of the timing model

A pulsar timing model predicts the rotational phase of the pulsar at any time.
For every observed TOA we compute the model phase; the difference between the
observed and predicted arrival time is the **timing residual**,

$$ R_i = t_i^{\rm obs} - t_i^{\rm model}. $$

Getting from a topocentric TOA to pulsar rotational phase involves a chain of
corrections — clock corrections, the Roemer and Shapiro delays in the solar
system, dispersion in the interstellar medium, and (for a binary) the orbital
delays at the pulsar's end:

$$ \Delta t = \Delta_{\rm clock} + \Delta_{\rm R\odot} + \Delta_{\rm S\odot}
 + \Delta_{\rm DM} + \Delta_{\rm R,bin} + \Delta_{\rm S,bin} + \ldots $$

A good model leaves residuals that are white, Gaussian scatter consistent with
the TOA uncertainties. **Structure in the residuals is information**: every
mis-set or missing parameter has a characteristic residual signature, and the
craft of timing is learning to read them. Today we will read one of the more
beautiful ones — the general-relativistic **Shapiro delay** of a binary
companion. (What to do when the leftover structure is *noise* rather than a
deterministic parameter is the subject of notebook 2.)


## 1. Warm-up: timing a pulsar with PINT

PINT ("PINT Is Not Tempo3") represents the timing model as a composable set of
Python objects, uses `astropy` units throughout, and is entirely independent of
`tempo`/`tempo2`. We start with the bundled example pulsar NGC 6440E.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from io import StringIO

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

import pint.logging
pint.logging.setup(level="WARNING")   # keep the notebook quiet; use INFO when working

import pint.config
import pint.fitter
from pint.models import get_model, get_model_and_toas
from pint.residuals import Residuals
from pint.simulation import make_fake_toas_uniform

In [ ]:
# PINT ships a set of example datasets
parfile = pint.config.examplefile("NGC6440E.par")
timfile = pint.config.examplefile("NGC6440E.tim")

# Load the timing model and the TOAs in one call
m, t = get_model_and_toas(parfile, timfile)

t.print_summary()

In [ ]:
# The model object knows its parameters, their uncertainties, and which are free
print(m)

In [ ]:
# Pre-fit residuals: model vs data before any fitting
rs = Residuals(t, m)

plt.figure(figsize=(9, 3.5))
plt.errorbar(t.get_mjds().value, rs.time_resids.to_value(u.us),
             yerr=t.get_errors().to_value(u.us), fmt="x")
plt.title(f"{m.PSR.value} pre-fit residuals")
plt.xlabel("MJD"); plt.ylabel("Residual ($\\mu$s)"); plt.grid(alpha=0.4)
plt.show()

There is clear structure — the par file parameters are close but not right.
The parameters flagged with a `1` in the par file (here F0, F1, RA, DEC, DM) are
free; `Fitter.auto` picks an appropriate fitter and does a weighted
least-squares fit.


In [ ]:
f = pint.fitter.Fitter.auto(t, m)
f.fit_toas()

print(f"Reduced chi^2 : {f.resids.chi2_reduced:.3f}")
print(f"Post-fit RMS  : {f.resids.time_resids.std().to(u.us):.3f}")

plt.figure(figsize=(9, 3.5))
plt.errorbar(t.get_mjds().value, f.resids.time_resids.to_value(u.us),
             yerr=t.get_errors().to_value(u.us), fmt="x")
plt.title(f"{m.PSR.value} post-fit residuals")
plt.xlabel("MJD"); plt.ylabel("Residual ($\\mu$s)"); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Compare pre- and post-fit parameter values and uncertainties
f.print_summary()

**Exercises (5 min)**

- Print `f.model.as_parfile()` — this is the updated par file you would save.
- Try excising the noisiest TOAs (`t.table["error"] > 30 * u.us`) and refit.
  Do the parameter uncertainties improve?
- What happens to the post-fit residuals if you freeze `F1`
  (`m.F1.frozen = True`) and refit?


## 2. Solving a Shapiro delay — weighing a neutron star with fake data

When a pulsar's radio beam passes close to its binary companion, the pulses are
delayed by the companion's spacetime curvature — the **Shapiro delay**. For a
companion of mass $M_2$ and an orbit inclined at angle $i$ to the sky, the extra
delay in a (nearly) circular orbit is

$$ \Delta_S(\Phi) = -2 r \, \ln\!\big[ 1 - s \sin \Phi \big], $$

where $\Phi$ is the orbital phase measured from the ascending node, and the two
**post-Keplerian parameters** are

$$ r = T_\odot M_2, \qquad s = \sin i, \qquad
   T_\odot = \frac{G M_\odot}{c^3} = 4.925\,\mu\mathrm{s}. $$

Two things make this delay precious:

1. $r$ and $s$ are *directly* the companion mass and inclination — combined with
   the Keplerian mass function they let you solve for the **pulsar mass**.
2. It is only measurable when the orbit is close to edge-on ($s \to 1$), because
   for low inclinations $\Delta_S$ is smooth and almost entirely **absorbed** by
   small adjustments to the Keplerian parameters ($x$, $T_{\rm asc}$, …). What
   survives in the residuals is only the sharp, high-harmonic spike near
   superior conjunction ($\Phi = 90^\circ$, orbital phase 0.25 past the
   ascending node).

Before diving in, spend two minutes with this
[interactive binary pulsar timing visualiser](https://mbailes.github.io/pulsar-timing/)
(by Matthew Bailes): pick a system, watch the orbit from the plane of the sky,
and see the Shapiro delay accumulate in real time alongside the Roemer and
Einstein delays as the pulsar swings behind its companion. Try changing the
companion mass and inclination — the same two knobs we are about to fit.

We will hide a Shapiro delay in simulated data and then find it.

### 2a. Simulate the "truth"

Our fake pulsar is loosely modelled on J1909−3744: a 2.95 ms pulsar in a
1.53-day, nearly edge-on orbit with a ~0.21 M$_\odot$ white-dwarf companion.


In [ ]:
TRUE_PAR = """
PSR              J1909-SIM
RAJ              19:09:47.4
DECJ             -37:44:14.5
POSEPOCH         59000
F0               339.315692    1
F1               -1.615e-15    1
PEPOCH           59000
DM               10.39
DMEPOCH          59000
BINARY           ELL1
PB               1.533449      1
A1               1.897991      1
TASC             59000.05      1
EPS1             4.5e-08       1
EPS2             -9.9e-08      1
M2               0.21
SINI             0.998
EPHEM            DE440
CLOCK            TT(BIPM2019)
UNITS            TDB
TZRMJD           59000
TZRSITE          meerkat
TZRFRQ           1284
"""

m_true = get_model(StringIO(TRUE_PAR))

# ~2 years of 0.5 us TOAs at MeerKAT. add_noise draws white noise at the TOA errors.
np.random.seed(42)
toas_sim = make_fake_toas_uniform(
    startMJD=59000, endMJD=59700, ntoas=500,
    model=m_true, obs="meerkat", freq=1284 * u.MHz,
    error=0.5 * u.us, add_noise=True,
)
print(f"Simulated {len(toas_sim)} TOAs")

# Save par/tim to disk — we will feed the same files to JUG in Part 3
import pathlib
datadir = pathlib.Path("fake_data"); datadir.mkdir(exist_ok=True)
toas_sim.write_TOA_file(datadir / "J1909-SIM.tim", format="tempo2")
(datadir / "J1909-SIM_true.par").write_text(TRUE_PAR)

### 2b. Fit *without* the Shapiro delay

Imagine we have just discovered this pulsar: we know its spin, position and
Keplerian orbit, but nothing about relativistic effects. Set $M_2 = 0$,
$\sin i = 0$ (no Shapiro delay in the model) and fit everything else.


In [ ]:
NOSHAP_PAR = TRUE_PAR.replace("M2               0.21",  "M2               0.0") \
                     .replace("SINI             0.998", "SINI             0.0")

f_noshap = pint.fitter.Fitter.auto(toas_sim, get_model(StringIO(NOSHAP_PAR)))
f_noshap.fit_toas()

res_noshap = f_noshap.resids.time_resids.to_value(u.us)
err_us = toas_sim.get_errors().to_value(u.us)
mjd = toas_sim.get_mjds().value

print(f"Post-fit RMS without Shapiro delay: {res_noshap.std():.3f} us "
      f"(TOA errors are 0.5 us)")

In [ ]:
# Orbital phase measured from the ascending node
pb   = float(f_noshap.model.PB.value)
tasc = float(f_noshap.model.TASC.value)
orb_phase = ((mjd - tasc) / pb) % 1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].errorbar(mjd, res_noshap, yerr=err_us, fmt=".", ms=3, alpha=0.5)
axes[0].set_xlabel("MJD"); axes[0].set_ylabel("Residual ($\\mu$s)")
axes[0].set_title("vs time: no obvious structure")

axes[1].errorbar(orb_phase, res_noshap, yerr=err_us, fmt=".", ms=3, alpha=0.5)
axes[1].axvline(0.25, color="crimson", ls="--", lw=1,
                label="superior conjunction")
axes[1].set_xlabel("Orbital phase (from ascending node)")
axes[1].set_title("vs orbital phase: there it is")
axes[1].legend()
for ax in axes: ax.grid(alpha=0.4)
plt.tight_layout(); plt.show()

Plotted against **time**, the residuals mostly look like slightly-too-large
noise — the RMS is more than double the TOA uncertainty, with a scattering of
unexplained outliers. Folded at the **orbital period**, a sharp spike appears at
orbital phase 0.25: every time the pulsar passes behind its companion, the
pulses arrive *late* relative to the delay-free model.

Note what the fit has already done: the *smooth* part of the Shapiro delay has
been absorbed into small biases of $x$, $T_{\rm asc}$, and the eccentricity
parameters. Only the un-absorbable harmonic content is left. We can show this
explicitly by comparing the full injected delay to what survives the fit:


In [ ]:
T_SUN_US = 4.925490947  # G Msun / c^3 in microseconds
M2_TRUE, SINI_TRUE = 0.21, 0.998

phi_fine = np.linspace(0, 1, 2000)
full_shapiro = -2 * T_SUN_US * M2_TRUE * np.log(
    1 - SINI_TRUE * np.sin(2 * np.pi * phi_fine))
full_shapiro -= full_shapiro.mean()

# What the fit could NOT absorb = binned residuals of the no-Shapiro fit
nbins = 50
bins = np.linspace(0, 1, nbins + 1)
idx = np.digitize(orb_phase, bins) - 1
binned = np.array([res_noshap[idx == k].mean() if np.any(idx == k) else np.nan
                   for k in range(nbins)])

plt.figure(figsize=(9, 4))
plt.plot(phi_fine, full_shapiro, "k-", lw=1.2,
         label="full injected Shapiro delay (mean-subtracted)")
plt.plot(bins[:-1] + 0.5 / nbins, binned, "o", ms=5, color="crimson",
         label="left in residuals after Keplerian fit")
plt.axvline(0.25, color="grey", ls=":", lw=1)
plt.xlabel("Orbital phase (from ascending node)")
plt.ylabel("Delay ($\\mu$s)")
plt.title("Most of the Shapiro delay hides inside the Keplerian fit")
plt.legend(); plt.grid(alpha=0.4)
plt.show()

### 2c. Fit the Shapiro delay and weigh the star

Now free $M_2$ and $\sin i$ (starting from deliberately wrong guesses) and
refit.


In [ ]:
SHAP_PAR = TRUE_PAR.replace("M2               0.21",  "M2               0.15  1") \
                   .replace("SINI             0.998", "SINI             0.99  1")
(datadir / "J1909-SIM_start.par").write_text(SHAP_PAR)   # JUG will start here too

f_shap = pint.fitter.Fitter.auto(toas_sim, get_model(StringIO(SHAP_PAR)))
f_shap.fit_toas()

res_shap = f_shap.resids.time_resids.to_value(u.us)
m2   = f_shap.model.M2.value;   m2_err   = f_shap.model.M2.uncertainty_value
sini = f_shap.model.SINI.value; sini_err = f_shap.model.SINI.uncertainty_value

print(f"Post-fit RMS with Shapiro delay: {res_shap.std():.3f} us")
print(f"M2   = {m2:.4f} +/- {m2_err:.4f} Msun   (truth: 0.21)")
print(f"SINI = {sini:.5f} +/- {sini_err:.5f}      (truth: 0.998)")
print(f"=> inclination i = {np.degrees(np.arcsin(sini)):.2f} deg")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
axes[0].errorbar(orb_phase, res_noshap, yerr=err_us, fmt=".", ms=3, alpha=0.5)
axes[0].set_title(f"Without Shapiro delay   (RMS = {res_noshap.std():.3f} $\\mu$s)")
axes[1].errorbar(orb_phase, res_shap, yerr=err_us, fmt=".", ms=3, alpha=0.5,
                 color="seagreen")
axes[1].set_title(f"With Shapiro delay   (RMS = {res_shap.std():.3f} $\\mu$s)")
axes[1].set_xlabel("Orbital phase (from ascending node)")
for ax in axes:
    ax.axvline(0.25, color="crimson", ls="--", lw=1)
    ax.set_ylabel("Residual ($\\mu$s)"); ax.grid(alpha=0.4)
plt.tight_layout(); plt.show()

The spike is gone, the RMS has dropped to the TOA uncertainty, and we have
measured the companion mass and inclination. The Keplerian **mass function**

$$ f(M_p) = \frac{(M_2 \sin i)^3}{(M_p + M_2)^2}
          = \frac{4\pi^2 x^3}{T_\odot P_b^2}, \qquad x = a_1 \sin i / c $$

then gives the **pulsar mass**:


In [ ]:
x  = float(f_shap.model.A1.value)          # projected semi-major axis (lt-s)
pb_s = float(f_shap.model.PB.value) * 86400  # orbital period (s)
T_SUN_S = 4.925490947e-6                     # seconds

fmass = 4 * np.pi**2 * x**3 / (T_SUN_S * pb_s**2)   # in Msun
mp = np.sqrt((m2 * sini)**3 / fmass) - m2

# First-order error propagation (dominated by M2)
dmp_dm2 = 1.5 * np.sqrt(m2 * sini**3 / fmass) - 1
mp_err = abs(dmp_dm2) * m2_err

print(f"Mass function f = {fmass:.6f} Msun")
print(f"Pulsar mass  Mp = {mp:.2f} +/- {mp_err:.2f} Msun")
print("\nYou have just weighed a neutron star.")

**Exercises (10 min)**

1. **Inclination matters.** Re-run the simulation with `SINI 0.7`
   ($i \approx 44^\circ$) and repeat 2b. Can you still see the signature at
   orbital phase 0.25? Can you still fit M2 meaningfully?
2. **Precision matters.** Restore `SINI 0.998` but degrade the TOA errors to
   5 $\mu$s. How do the M2 and pulsar-mass uncertainties respond?
3. **Orthometric parameterisation.** For marginal detections the $(r, s)$
   parameters are strongly correlated, and the *orthometric* parameters
   $(h_3, \varsigma)$ of Freire & Wex (2010) are better behaved. Replace
   `M2/SINI` with `H3`/`STIG` (binary model `ELL1H`) and compare the
   correlation of the fitted parameters.


## 3. The same fit in JUG

JUG (*JAX-based Unified pulsar timinG*) is an independent timing package that
compiles its residual and derivative calculations with JAX, so fits run on CPU
or GPU at speed even with tens of thousands of TOAs and a full noise model. The
workflow revolves around a single `TimingSession` object.

It reads the exact same par/tim files we just wrote with PINT.


In [ ]:
# JAX must be configured BEFORE it is first imported:
#  - run on CPU (robust everywhere; change to "cuda" if your CUDA/cuDNN stack
#    is healthy and you want the GPU)
#  - use 64-bit floats (pulsar timing needs the precision)
# If this cell raises "JAX already imported", do Kernel -> Restart and run the
# notebook again from the top: JAX fixes its compute platform at first import,
# so this setting cannot be changed afterwards.
import os, sys
if "jax" in sys.modules:
    raise RuntimeError("JAX already imported — restart the kernel and rerun from the top")
os.environ["JAX_PLATFORMS"] = "cpu"

from jug.utils.jax_setup import ensure_jax_x64
ensure_jax_x64()

from jug.engine.session import TimingSession

session = TimingSession(datadir / "J1909-SIM_start.par",
                        datadir / "J1909-SIM.tim", verbose=False)

# inspect() lists everything the session can do
session.inspect()

In [ ]:
session.print_model()

In [ ]:
# Pre-fit residuals
result = session.compute_residuals(subtract_tzr=True)
prefit_us = result["residuals_us"]
print(f"JUG pre-fit weighted RMS: {session.weighted_rms(prefit_us):.4f} us")

In [ ]:
import time

t0 = time.time()
fit_jug = session.fit_parameters(max_iter=10, verbose=False)
print(f"Fit wall time: {time.time() - t0:.2f} s "
      f"({fit_jug['iterations']} iterations, converged={fit_jug['converged']})")
print(f"JUG post-fit weighted RMS: "
      f"{session.weighted_rms(fit_jug['residuals_us']):.4f} us")

In [ ]:
# Side-by-side comparison of the Shapiro parameters
jug_m2,  jug_m2e  = fit_jug["final_params"]["M2"],   fit_jug["uncertainties"]["M2"]
jug_sini, jug_sinie = fit_jug["final_params"]["SINI"], fit_jug["uncertainties"]["SINI"]

print(f"{'':10s}{'PINT':>22s}{'JUG':>22s}{'truth':>10s}")
print(f"{'M2':10s}{m2:>12.5f} ± {m2_err:<8.5f}{jug_m2:>12.5f} ± {jug_m2e:<8.5f}{0.21:>10.3f}")
print(f"{'SINI':10s}{sini:>12.6f} ± {sini_err:<7.6f}{jug_sini:>12.6f} ± {jug_sinie:<7.6f}{0.998:>10.3f}")

In [ ]:
# Full pre-fit vs post-fit parameter table
session.parameter_table(fit_jug)

Two completely independent codes — different ephemeris handling, different
fitters, different languages under the hood — agree on the Shapiro parameters to
well within 1σ. Cross-validation between independent packages is exactly how
real PTA collaborations build confidence in their measurements.

JUG also ships an interactive GUI (in the spirit of `plk`/`pintk`) which is
great for exploratory timing — box-zoom, box-delete, toggling noise processes:

```bash
jug-gui --par fake_data/J1909-SIM_start.par --tim fake_data/J1909-SIM.tim
```

**Exercise:** use `session.set_free("F2")` and refit with
`session.fit_parameters(fit_params=["F2"])`. Is the recovered F2 consistent
with zero (nothing was injected)?


## 4. A real one: J1614−2230, the two-solar-mass pulsar

Time to do this on real data. **PSR J1614−2230** is the pulsar whose Shapiro
delay measurement (Demorest et al. 2010, *Nature*) delivered the first
~2 M$_\odot$ neutron star — a result that instantly ruled out large families
of soft equations of state for nuclear matter. The orbit is almost perfectly
edge-on ($i \approx 89.2°$) with a massive white-dwarf companion
($M_2 \approx 0.5\,$M$_\odot$), so the Shapiro delay is enormous: the full
cusp spans over 30 $\mu$s.

The NANOGrav 12.5-year **wideband** par/tim files ship with PINT (wideband
TOAs carry a simultaneous DM measurement; `Fitter.auto` quietly picks the
wideband fitter — the API is unchanged). The binary model is ELL1, so orbital
phase is measured from the ascending node exactly as in Part 2.

We reproduce the logic of the famous Demorest et al. figure in three panels:

- **(a)** take the full fitted solution and simply *switch off* the Shapiro
  delay without refitting — the residuals trace the entire cusp;
- **(b)** refit all Keplerian parameters with M2/SINI held at zero — the
  smooth part gets absorbed and only the sharp unabsorbable signature
  survives (Part 2b on real data);
- **(c)** the full fit.

The two fits take ~20 s.


In [ ]:
import copy

m_j1614, t_j1614 = get_model_and_toas(
    pint.config.examplefile("J1614-2230_NANOGrav_12yv3.wb.gls.par"),
    pint.config.examplefile("J1614-2230_NANOGrav_12yv3.wb.tim"))

err_j = t_j1614.get_errors().to_value(u.us)
phase_j = ((t_j1614.get_mjds().value - float(m_j1614.TASC.value))
           / float(m_j1614.PB.value)) % 1.0

def fit_j1614(with_shapiro):
    mm = copy.deepcopy(m_j1614)
    if not with_shapiro:
        mm.M2.value, mm.SINI.value = 0.0, 0.0
        mm.M2.frozen, mm.SINI.frozen = True, True
    f = pint.fitter.Fitter.auto(t_j1614, mm)
    f.fit_toas()
    # wideband residuals live under .toa (the .dm part holds the DM residuals)
    res = f.resids.toa.time_resids.to_value(u.us)
    return f, res - np.average(res, weights=1 / err_j**2)

f_nos_j,  res_nos_j  = fit_j1614(with_shapiro=False)
f_full_j, res_full_j = fit_j1614(with_shapiro=True)

m2_j   = f_full_j.model.M2.value
sini_j = f_full_j.model.SINI.value
print(f"Fitted:    M2 = {m2_j:.4f} +/- {f_full_j.model.M2.uncertainty_value:.4f} Msun, "
      f"SINI = {sini_j:.6f}")
print(f"Published: M2 = {m_j1614.M2.quantity}, SINI = {m_j1614.SINI.value}")

# Panel (a): full solution with the Shapiro delay switched off, NO refit
m_frozen = copy.deepcopy(f_full_j.model)
m_frozen.M2.value, m_frozen.SINI.value = 0.0, 0.0
res_zero = Residuals(t_j1614, m_frozen).time_resids.to_value(u.us)
res_zero -= np.average(res_zero, weights=1 / err_j**2)

In [ ]:
# Theory curve for panel (a): the full ELL1 Shapiro delay for the fitted M2, sin i.
# Subtract its TOA-weighted mean (the TOAs cluster near conjunction, so the
# uniform-grid mean would sit visibly off the data).
phi_grid = np.linspace(0.001, 0.999, 2000)
shap = lambda phi: -2 * T_SUN_US * m2_j * np.log(1 - sini_j * np.sin(2 * np.pi * phi))
offset = np.average(shap(phase_j), weights=1 / err_j**2)

fig, axes = plt.subplots(3, 1, figsize=(9.5, 8.5), sharex=True)

axes[0].errorbar(phase_j, res_zero, yerr=err_j, fmt=".", ms=3.5, alpha=0.6,
                 color="steelblue")
axes[0].plot(phi_grid, shap(phi_grid) - offset, "-", color="crimson", lw=1.5,
             label="full Shapiro delay for fitted $M_2$, $\\sin i$")
axes[0].set_title("(a) full solution, Shapiro delay switched off — no refit")
axes[0].legend(loc="upper right")

axes[1].errorbar(phase_j, res_nos_j, yerr=err_j, fmt=".", ms=3.5, alpha=0.6,
                 color="steelblue")
axes[1].set_title("(b) Keplerian refit without the Shapiro delay — only the\n"
                  "unabsorbable signature survives")

axes[2].errorbar(phase_j, res_full_j, yerr=err_j, fmt=".", ms=3.5, alpha=0.6,
                 color="seagreen")
axes[2].set_title(f"(c) full fit (RMS = {res_full_j.std():.2f} $\\mu$s)")
axes[2].set_xlabel("Orbital phase (from ascending node)")

for ax in axes:
    ax.axvline(0.25, color="grey", ls=":", lw=1)
    ax.set_ylabel("Residual ($\\mu$s)"); ax.grid(alpha=0.35)
plt.tight_layout(); plt.show()

In [ ]:
# And the payoff: weigh the pulsar
x_j    = float(f_full_j.model.A1.value)
pb_s_j = float(f_full_j.model.PB.value) * 86400

fmass_j = 4 * np.pi**2 * x_j**3 / (T_SUN_S * pb_s_j**2)
mp_j = np.sqrt((m2_j * sini_j)**3 / fmass_j) - m2_j
print(f"Pulsar mass Mp = {mp_j:.2f} Msun   (Demorest et al. 2010: 1.97 +/- 0.04)")

Panel (a) is one of the great figures of modern astrophysics reproduced from
public data in a few lines: the residuals trace the full relativistic cusp.
Panel (b) shows the honest version — what a timer who does not believe in
general relativity can actually see — and panel (c) shows the delay fully
modelled, leaving 0.26 $\mu$s residuals over 12.5 years.

A convention footnote: if you try this with a pulsar using the **DD** binary
model (e.g. B1855+09, also bundled with PINT), remember that DD's epoch T0 is
the time of *periastron*, not the ascending node — fold naively from T0 and
the conjunction spike lands at $(90° - \omega)/360$, not 0.25. For a
near-circular orbit, shift the fold by $\omega/360$ to recover this notebook's
convention.

**Exercise:** the fit above also measures $\dot{P}_b$-free Keplerian and
astrometric parameters. Compute the characteristic age
$\tau_c = P/(2\dot{P})$, surface field
$B_s \simeq 10^{12}\,\mathrm{G}\,(\dot{P}/10^{-15})^{1/2}(P/\mathrm{s})^{1/2}$
and spin-down luminosity
$\dot{E} = 4\pi^2 I \dot{P}/P^3$ (with $I = 10^{45}\,\mathrm{g\,cm^2}$) from
the fitted F0 and F1. Where would this pulsar sit on a $P$–$\dot{P}$ diagram
relative to the normal-pulsar population?


## 5. Interactive timing with `jug-gui`

Real timing work is iterative and visual: you stare at residuals, toggle
parameters, delete a corrupted TOA, fit again. JUG ships a fast interactive
GUI (in the spirit of tempo2's `plk` and PINT's `pintk`) for exactly this
loop. Launch it on the simulated Shapiro dataset from Part 2 — run this in a
terminal, not in the notebook:

```bash
jug-gui --par fake_data/J1909-SIM_start.par --tim fake_data/J1909-SIM.tim
```

### A guided exploration

1. **Look at the pre-fit residuals.** The y-axis selector (top left) shows
   *Pre-fit (µs)*; the x-axis selector offers MJD, Serial, Frequency, TOA
   error, **Orbital Phase**, Day of Year, and Year. Switch the x-axis to
   *Orbital Phase* — the par file starts with the wrong M2/SINI, so you should
   recognise part of the Shapiro signature from Part 2b immediately.
2. **Choose what to fit.** Open the parameter drawer (`Ctrl+E`) and tick or
   untick parameters — this is the GUI equivalent of `set_free`/`set_frozen`.
   Try fitting once *without* M2/SINI, and once with them.
3. **Fit** with `Ctrl+F`. The y-axis flips to *Post-fit (µs)*. Refold in
   orbital phase and watch the spike disappear when the Shapiro parameters are
   included. `Ctrl+R` restarts from the original par file whenever you have
   painted yourself into a corner.
4. **Inspect and clean.** Drag a box with `Z` to zoom into a cluster of
   points; `U` unzooms. Drag a box with `Shift+Z` to *delete* the TOAs
   inside it — then refit and watch the parameter uncertainties respond.
   `Ctrl+Shift+A` restores every deleted TOA.
5. **Epoch-average** the display with `Ctrl+A` (like tempo2's averaging) to
   see structure hiding under dense multi-frequency epochs.
6. **Save your work**: `Ctrl+S` writes the fitted par file, `Ctrl+Shift+S`
   the (possibly cleaned) tim file.

When you load a pulsar whose par file carries a noise model (like the
red-noise dataset in the second notebook), two more tricks become available:

- **Show Noise Panel** (View menu) lists the noise processes in the model and
  lets you toggle them individually;
- `Shift+K` subtracts the fitted **red-noise realisation** from the displayed
  residuals, and `Shift+F` does the same for the **DM-noise realisation** —
  the interactive version of the whitening you will do in notebook 2.

### Command reference

| Action | Shortcut |
|---|---|
| Open par / tim file | `Ctrl+P` / `Ctrl+T` |
| Parameter drawer (choose fit parameters) | `Ctrl+E` |
| Run fit | `Ctrl+F` |
| Restart from original par | `Ctrl+R` |
| Box zoom (drag) / unzoom / zoom to fit | `Z` / `U` / `Ctrl+0` |
| Box delete TOAs (drag) | `Shift+Z` |
| Restore all deleted TOAs | `Ctrl+Shift+A` |
| Epoch-average TOAs (display) | `Ctrl+A` |
| Subtract red-noise realisation | `Shift+K` |
| Subtract DM-noise realisation | `Shift+F` |
| Toggle residual display | `R` |
| Save par / tim / pulse-numbered tim | `Ctrl+S` / `Ctrl+Shift+S` / `Ctrl+N` |
| Quit | `Ctrl+Q` |

The **View** menu also has *Show Noise Panel*, *Show Zero Line*, backend
colouring (*Color By*), and light/dark themes.

**Exercise:** using only the GUI, delete the five highest-uncertainty TOAs,
refit, and compare the M2 uncertainty against the notebook value from Part 3.


## Wrap-up

In this notebook you have:

- fitted timing models in two independent packages, **PINT** and **JUG**, and
  checked they agree to well within 1σ;
- simulated realistic TOAs, hidden a relativistic effect in them, recovered
  it, and **measured a neutron-star mass** from it;
- seen why an unmodelled Shapiro delay (or any smooth delay) partially hides
  inside the Keplerian parameters, and what residual signature survives;
- reproduced the **J1614−2230 two-solar-mass measurement** from public data;
- driven the whole loop interactively with `jug-gui`.

**Where to go next**

- Notebook 2 (`timing_2_noise_gaussianity.ipynb`): red and DM noise,
  noise realisations, whitening, and Gaussianity testing.
- [PINT example: simulate a relativistic binary and build a mass–mass
  diagram](https://nanograv-pint.readthedocs.io/en/latest/examples/Simulate_and_make_MassMass.html)
  — extend today's single Shapiro measurement to a multi-parameter GR test.
- [PINT example: checking phase
  connection](https://nanograv-pint.readthedocs.io/en/latest/examples/check_phase_connection.html)
  — how a timing solution is built from scratch when you *don't* already have
  a good par file (the core craft of real-world timing).
- The [interactive binary visualiser](https://mbailes.github.io/pulsar-timing/)
  from Part 2, for building intuition about every delay term at once.
